# k02 — The event stream is the contract

Every finstack-ai surface — this notebook, the `finstack-know` CLI's text
and `--json` renderers, the browser example — consumes the **same**
`RunEventKind` stream. This notebook watches it directly.

Trust: T2 callback. Network: none (scripted model).

In [ ]:
import asyncio
import tempfile
from collections import Counter
from pathlib import Path

from _knowledge import build_knowledge_agent, scripted_model

workdir = Path(tempfile.mkdtemp(prefix="finstack-know-k02-"))
agent = await build_knowledge_agent(
    workdir,
    scripted_model(
        ["The six ports are model, tool, context, middleware, observer, journal."]
    ),
)

## Stream a run's event batches

`Agent.start` returns a `Run` handle; `run.events()` is an
`EventBatchIterator`. Batches arrive while the run executes — the CLI
feeds exactly this iterator into its renderers. Tabulating the kinds
shows the run's anatomy: acceptance, model deltas, message finalization,
completion.

In [ ]:
run = agent.start("What are the six runtime ports?")


async def _collect(handle):
    kinds = []
    async for batch in handle.events():
        kinds.extend(event.kind for event in batch.events())
    return kinds


result, kinds = await asyncio.gather(run.result(), _collect(run))
print(result.text)
print(Counter(kinds))
for expected in ("message_finalized", "run_completed"):
    assert expected in kinds, kinds

## The committed trace and active capabilities

`RunResult.trace` is the Rust-owned list of committed record kinds in
journal order — the durable spine the events were derived from.
`active_capabilities` lists which declared capabilities were active for
the run (the citations skill activates only when the model asks for it,
so it is absent here).

In [ ]:
print(result.trace)
print(result.active_capabilities)
assert result.trace, "committed records exist"
capability_ids = [entry["id"] for entry in result.active_capabilities]
assert "finstack.know.skill.citations" not in capability_ids

## Same stream, other surfaces

Run the CLI with `--json` against any data directory and compare:

```bash
finstack-know --data-dir /tmp/kd ask "hello" --json
```

Each NDJSON line is one of these events with its kind name verbatim. An
event kind observed on one surface and absent from another is a failing
conformance test (`test_knowledge_golden.py` here, `golden.rs` in the app
crate, the Playwright spec in the browser example).

In [ ]:
import shutil

shutil.rmtree(workdir, ignore_errors=True)
print("cleaned", workdir)